# Decision Model Demo — Entity Resolution & Decision-Model-as-Judge

Live demo for the Decision Models deep-dive session (Edition #1).

Two scenarios, one running model:
1. **Entity resolution** — same entity, yes/no + calibrated probability.
2. **Decision-model-as-judge** — same server, swapped question schema, judging a RAG answer against a rubric.

Then a **contrast**: the same entity-resolution question answered by prompting a generic LLM for JSON, timed side by side.

Model: [Torchcast Decision 4B](https://github.com/Torchcast-AI/torchcast-decision-4b) (Apache-2.0, built on Qwen3.5-4B), served locally behind its TypeSafe-compatible `/v1/systemone` endpoint.

## Setup — run this **before** the session, not live

```bash
python3.12 -m venv .venv && source .venv/bin/activate
pip install "torch==2.9.1" --index-url https://download.pytorch.org/whl/cu128
pip install "transformers==5.8.1" "flash-linear-attention==0.5.2" accelerate safetensors "huggingface_hub[cli]" ninja packaging wheel setuptools
pip install --no-build-isolation "causal-conv1d>=1.7"

hf download torchcast-ai/torchcast-decision-4b --revision v1.3.1 --local-dir torchcast-decision-4b-v1.3
cd torchcast-decision-4b-v1.3 && sha256sum -c SHA256SUMS

python -m torchcast_decision.check .
python -m torchcast_decision.server --model . --host 127.0.0.1 --port 8011
```

On Apple Silicon, swap in the MLX 8-bit build (`arinltte/torchcast-decision-4B-MLX-8bit`) for faster local inference without a GPU.

Leave the server running, then execute the cells below.

In [ ]:
import json
import time

import requests

SERVER_URL = "http://127.0.0.1:8011"

# If the real server from the Setup cell isn't reachable (e.g. drafting this
# notebook without a GPU handy), SIMULATED mode returns a canned response so
# the rest of the notebook still runs end to end. The live session must run
# against the real server -- SIMULATED output is clearly labeled below.
try:
    requests.get(f"{SERVER_URL}/health", timeout=2).raise_for_status()
    SIMULATED = False
except requests.RequestException:
    SIMULATED = True

print("SIMULATED mode (no live server found)" if SIMULATED else "Connected to live Torchcast Decision server")

In [ ]:
def decide(state: str, questions: dict) -> tuple[dict, float]:
    """POST to the TypeSafe-compatible /v1/systemone endpoint. Returns (response_json, elapsed_seconds)."""
    payload = {"state": state, "questions": questions}
    start = time.perf_counter()

    if SIMULATED:
        time.sleep(0.18)  # representative of the 70-500ms range for a warm small open decision model
        result = {
            "_simulated": True,
            "decision": {"noul": {"yes": 0.93, "no": 0.07}},
        }
    else:
        response = requests.post(f"{SERVER_URL}/v1/systemone", json=payload, timeout=10)
        response.raise_for_status()
        result = response.json()

    elapsed = time.perf_counter() - start
    return result, elapsed

## Scenario 1 — Entity resolution

State = two candidate customer/counterparty records. Question = "same entity: yes/no."

In [ ]:
record_a = {"name": "Jon A. Smith", "address": "221B Baker St, London", "dob": "1985-03-12", "id_last4": "4471"}
record_b = {"name": "Jonathan Smith", "address": "221 B Baker Street, London", "dob": "1985-03-12", "id_last4": "4471"}

entity_state = (
    "Candidate record A: " + json.dumps(record_a) + "\n"
    "Candidate record B: " + json.dumps(record_b)
)

entity_questions = {
    "decision": {
        "type": "yes_no",
        "instructions": "Do candidate record A and candidate record B refer to the same real-world entity?",
    }
}

entity_result, entity_elapsed = decide(entity_state, entity_questions)
print(f"elapsed: {entity_elapsed*1000:.0f} ms")
print(json.dumps(entity_result, indent=2))

## Scenario 2 — Decision-model-as-judge

Same model, same server, a different question schema: judging a RAG answer against a pass/fail rubric instead of matching entities.

In [ ]:
judge_state = json.dumps({
    "question": "What is our policy on reporting transactions above the regulatory threshold?",
    "retrieved_context": (
        "Per internal policy RM-114, transactions exceeding the regulatory threshold "
        "must be reported to the compliance desk within 24 hours of detection."
    ),
    "generated_answer": (
        "Transactions above the regulatory threshold must be reported to the "
        "compliance desk within 24 hours, per policy RM-114."
    ),
})

judge_questions = {
    "decision": {
        "type": "yes_no",
        "instructions": (
            "Does the generated_answer accurately and completely answer the question, "
            "using only information supported by retrieved_context? Answer pass (yes) or fail (no)."
        ),
    }
}

judge_result, judge_elapsed = decide(judge_state, judge_questions)
print(f"elapsed: {judge_elapsed*1000:.0f} ms")
print(json.dumps(judge_result, indent=2))

## Contrast — the same entity-resolution question via a generic LLM prompted for JSON

Wire `call_llm_for_json` below to whatever LLM endpoint is already approved for use on this team (e.g. an internal gateway to Claude or another provider). Left unimplemented here deliberately: which client/credentials are available depends on org access this notebook doesn't have, so it's a presenter-side fill-in rather than a guess.

In [ ]:
def call_llm_for_json(prompt: str) -> tuple[dict, float]:
    """TODO before the live session: wire this to the team's approved LLM client.
    Must return (parsed_json_response, elapsed_seconds).
    """
    raise NotImplementedError("Wire up an LLM client here before running live.")


llm_prompt = (
    "Given these two records, answer with ONLY a JSON object "
    '{"same_entity": true|false, "confidence": <0-1 float>}.\n\n'
    f"Record A: {json.dumps(record_a)}\nRecord B: {json.dumps(record_b)}"
)

# llm_result, llm_elapsed = call_llm_for_json(llm_prompt)
# print(f"elapsed: {llm_elapsed*1000:.0f} ms")
# print(json.dumps(llm_result, indent=2))
#
# print("\n--- Timing comparison ---")
# print(f"Decision model (entity resolution): {entity_elapsed*1000:.0f} ms")
# print(f"Generic LLM prompted for JSON:       {llm_elapsed*1000:.0f} ms")

## Discussion prompt

Where in our stack could a decision model replace an LLM-as-judge call or a hand-rolled classifier?